# Data Collection and Pre-Processing Lab

This notebook executes the 12-step data-engineering road map on **500 synthetic e-commerce transactions created for this assignment**. The source CSV is deliberately imperfect so the cleaning work can be measured. Reusable logic lives in `src/data_pipeline.py`; this notebook imports and calls it. The raw file remains unchanged.


## Step 1 — Hello, Data!

Load the raw CSV and display the first three rows. `PROJECT_ROOT` points one level above this `notebooks/` folder, so the same relative structure works after the repository is cloned.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Enables a plain-Python verification outside Jupyter.

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_pipeline import (
    Transaction,
    add_transformations,
    clean_data,
    dataframe_to_records,
    engineer_features,
    load_data,
    profile_data,
    quality_counts,
    revenue_by_city,
    serialize_data,
)

RAW_FILE = PROJECT_ROOT / "data" / "ecommerce_transactions_raw.csv"
CITY_FILE = PROJECT_ROOT / "data" / "city_metadata.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

raw = load_data(RAW_FILE)
assert len(raw) == 500, "The assignment dataset must contain exactly 500 rows."
print("Raw shape:", raw.shape)
display(raw.head(3))


Raw shape: (500, 8)


,transaction_id,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,TXN00001,2024-01-13,CUST0071,Mechanical Keyboard,85.4,2,SAVE15,Montreal
1,TXN00002,2024-02-14,CUST0152,Fitness Tracker,83.43,1,NaN,Ottawa-Gatineau
2,TXN00003,2024-09-15,CUST0155,Laptop Stand,42.78,2,SAVE15,Hamilton


## Step 2 — Pick the Right Container

A **dictionary** is appropriate for one flexible row because every value has a named key. A dataclass gives one transaction a consistent structure plus methods such as `.clean()` and `.total()`. A **set** is best for unique city values, but it cannot preserve repeated transactions. pandas remains the main container for bulk tabular work.


In [2]:
first_row_dictionary = raw.iloc[0].to_dict()
unique_cities = set(raw["shipping_city"].dropna())

print("Dictionary keys:", list(first_row_dictionary))
print("Raw city values in the set:", len(unique_cities))


Dictionary keys: ['transaction_id', 'date', 'customer_id', 'product', 'price', 'quantity', 'coupon_code', 'shipping_city']
Raw city values in the set: 11


## Step 3 — Implement Functions and Data Structure

The `Transaction` dataclass is defined in `src/data_pipeline.py`. `from_dict()` populates it from a row dictionary, `.clean()` standardizes its text, and `.total()` calculates the discounted line total.


In [3]:
transaction = Transaction.from_dict(first_row_dictionary)
transaction.clean()

print("Transaction object:", transaction)
print("Discounted line total:", transaction.total())


Transaction object: Transaction(transaction_id='TXN00001', date='2024-01-13', customer_id='CUST0071', product='Mechanical Keyboard', price='85.4', quantity=2, coupon_code='SAVE15', shipping_city='Montreal')
Discounted line total: 145.18


## Step 4 — Bulk Loaded

The imported `dataframe_to_records()` function maps the DataFrame into a list of dictionaries. This is useful when another function or service expects ordinary Python objects rather than a DataFrame.


In [4]:
transaction_records = dataframe_to_records(raw)
print("Dictionary records:", len(transaction_records))
display(transaction_records[:2])


Dictionary records: 500


[{'transaction_id': 'TXN00001',
  'date': '2024-01-13',
  'customer_id': 'CUST0071',
  'product': 'Mechanical Keyboard',
  'price': '85.4',
  'quantity': 2,
  'coupon_code': 'SAVE15',
  'shipping_city': 'Montreal'},
 {'transaction_id': 'TXN00002',
  'date': '2024-02-14',
  'customer_id': 'CUST0152',
  'product': 'Fitness Tracker',
  'price': '83.43',
  'quantity': 1,
  'coupon_code': nan,
  'shipping_city': 'Ottawa-Gatineau'}]

## Step 5 — Quick Profiling

The profiling function converts price values safely, then reports minimum, mean and maximum price. It also uses a set internally to count distinct nonblank shipping cities.


In [5]:
profile = profile_data(raw)
display(pd.Series(profile, name="value"))


minimum_price         21.630000
mean_price            73.387255
maximum_price        174.540000
unique_city_count     11.000000
Name: value, dtype: float64

## Step 6 — Spot the Grime

The generator intentionally inserts several realistic problems: an impossible date, a text price, a negative quantity, a missing city, an unknown coupon, duplicate transaction ID and inconsistent text spacing/case. `quality_counts()` measures each category without changing the source.


In [6]:
before_counts = quality_counts(raw)
display(pd.Series(before_counts, name="before_cleaning"))


invalid_date                1
invalid_price               1
invalid_quantity            1
missing_shipping_city       1
unknown_coupon              1
duplicate_transaction_id    1
unclean_text                2
Name: before_cleaning, dtype: int64

## Step 7 — Cleaning Rules

`clean_data()` works on a copy. It parses dates and numeric fields, standardizes product/city/coupon text, treats unknown coupons as no coupon, removes unrecoverable rows, and keeps the first occurrence of a duplicated transaction ID. The raw CSV and `raw` DataFrame are not overwritten.


In [7]:
cleaned = clean_data(raw)
after_counts = quality_counts(cleaned)

comparison = pd.DataFrame({"before": before_counts, "after": after_counts})
display(comparison)
print(f"Rows retained: {len(cleaned)} of {len(raw)}")
assert all(value == 0 for value in after_counts.values())


,before,after
invalid_date,1,0
invalid_price,1,0
invalid_quantity,1,0
missing_shipping_city,1,0
unknown_coupon,1,0
duplicate_transaction_id,1,0
unclean_text,2,0


Rows retained: 495 of 500


## Step 8 — Transformations

Coupon codes map to numeric discount percentages. The second source, `city_metadata.csv`, adds province and 2021 census-metropolitan-area population by shipping city. `validate='many_to_one'` inside the function ensures each city has only one metadata row.


In [8]:
city_metadata = pd.read_csv(CITY_FILE)
transformed = add_transformations(cleaned, city_metadata)

print("Rows missing city metadata:", transformed["province"].isna().sum())
display(transformed[["coupon_code", "discount_pct", "shipping_city", "province", "cma_population_2021"]].head())


Rows missing city metadata: 0


,coupon_code,discount_pct,shipping_city,province,cma_population_2021
0,SAVE15,15,Montreal,Quebec,4291732
1,,0,Ottawa-Gatineau,Ontario/Quebec,1488307
2,SAVE15,15,Hamilton,Ontario,785184
3,WELCOME20,20,Vancouver,British Columbia,2642825
4,,0,Montreal,Quebec,4291732


## Step 9 — Feature Engineering

`days_since_purchase` uses the latest valid transaction date in this dataset as a fixed reference, so results do not change tomorrow. `revenue` equals `price × quantity × (1 − discount_pct/100)`.


In [9]:
featured = engineer_features(transformed)
reference_date = featured["date"].max()

print("Reference date:", reference_date.date())
display(featured[["date", "days_since_purchase", "price", "quantity", "discount_pct", "revenue"]].head())


Reference date: 2024-12-30


,date,days_since_purchase,price,quantity,discount_pct,revenue
0,2024-01-13,352,85.40,2,15,145.18
1,2024-02-14,320,83.43,1,0,83.43
2,2024-09-15,106,42.78,2,15,72.73
3,2024-04-22,252,121.63,1,20,97.30
4,2024-06-23,190,111.27,3,0,333.81


## Step 10 — Mini-Aggregation

Group discounted transaction revenue by shipping city and sort from highest to lowest. A dictionary version is also created to demonstrate an appropriate plain-Python structure.


In [10]:
city_revenue = revenue_by_city(featured)
revenue_dictionary = dict(zip(city_revenue["shipping_city"], city_revenue["revenue"]))

display(city_revenue)
print("Highest-revenue city in this sample:", city_revenue.iloc[0]["shipping_city"])


,shipping_city,revenue
0,Quebec City,12246.01
1,Calgary,11912.93
2,Hamilton,11757.66
3,Montreal,11418.69
4,Vancouver,10097.58
5,Edmonton,9853.61
6,Toronto,9595.38
7,Winnipeg,8470.05
8,Ottawa-Gatineau,7757.12
9,Kitchener-Cambridge-Waterloo,7544.24


Highest-revenue city in this sample: Quebec City


## Step 11 — Serialization Checkpoint

The imported serialization function writes the final dataset to **CSV and JSON**. Reading both files back verifies that serialization preserved the number of rows.


In [11]:
csv_path, json_path = serialize_data(featured, OUTPUT_DIR)

csv_rows = len(pd.read_csv(csv_path))
json_rows = len(json.loads(json_path.read_text(encoding="utf-8")))
assert csv_rows == json_rows == len(featured)

print("CSV:", csv_path)
print("JSON:", json_path)
print("Rows saved in each format:", csv_rows)


CSV: c:\Users\Lenovo\Documents\Machine learning programming\Assignments\ecommerce_data_engineering_lab\outputs\cleaned_transactions.csv
JSON: c:\Users\Lenovo\Documents\Machine learning programming\Assignments\ecommerce_data_engineering_lab\outputs\cleaned_transactions.json
Rows saved in each format: 495


## Step 12 — Soft Interview Reflection

Functions helped me separate the pipeline into small jobs that can be tested and reused. For example, `quality_counts()` measures problems, `clean_data()` fixes them, and `engineer_features()` adds analytical fields. The notebook therefore shows the workflow without repeating its internal logic. The `Transaction` class combines one record's data with behaviours such as cleaning and calculating its total. If a rule changes, I can update one function in the Python module instead of editing many notebook cells. This structure also makes errors easier to locate because every function has one clear responsibility.


## Data-Dictionary

The primary definitions describe the synthetic transaction header. The secondary definitions come from the small Statistics Canada metropolitan population extract. Derived fields state their exact creation rules.

| Field | Type after processing | Description | Source |
|---|---|---|---|
| transaction_id | string | Unique synthetic transaction identifier | Primary synthetic CSV |
| date | date | Purchase date; invalid values removed | Primary synthetic CSV |
| customer_id | string | Reusable synthetic customer identifier | Primary synthetic CSV |
| product | string | Product name; whitespace standardized | Primary synthetic CSV |
| price | float | Unit price in Canadian dollars | Primary synthetic CSV |
| quantity | integer | Number of items purchased | Primary synthetic CSV |
| coupon_code | string | Optional promotion code; unknown codes cleared | Primary synthetic CSV |
| shipping_city | string | Destination metropolitan area; text standardized | Primary synthetic CSV |
| province | string | Province associated with the metropolitan area | Secondary metadata CSV |
| cma_population_2021 | integer | 2021 census metropolitan area population | Secondary metadata CSV / Statistics Canada |
| discount_pct | integer | Coupon mapped to 0, 5, 10, 15 or 20 percent | Derived by `add_transformations()` |
| days_since_purchase | integer | Latest dataset date minus purchase date | Derived by `engineer_features()` |
| revenue | float | Price × quantity after coupon discount | Derived by `engineer_features()` |

Secondary source: [Statistics Canada — Population, Canada at a Glance](https://www150.statcan.gc.ca/n1/pub/12-581-x/2022001/sec1-eng.htm).


## Concise Analytical Insight

Quebec City produced the highest discounted revenue in this reproducible sample at **$12,246.01**, while Kitchener-Cambridge-Waterloo produced the lowest at **$7,544.24**. Because the transactions are synthetic, the comparison demonstrates the analysis method rather than real consumer behaviour. Population metadata provides context but does not prove that population caused the revenue difference.
